# E5 — Pipeline phương pháp đề xuất (Full Method)

E5 kết hợp toàn bộ 3 thành phần cải tiến:
1. **Mô hình E2 Data Augmentation** (`experiments/E2_augmentation/augmentation_b16_seed42/weights/best.pt`).
2. **Tiền xử lý ảnh thiếu sáng E3 (CLAHE)** cho các ảnh vùng tối.
3. **Suy luận theo vùng E4 (Tiled Inference)** với tile 640×640 và overlap 25% cho các vật thể nhỏ.

Thí nghiệm này nhằm đánh giá hiệu quả tổng thể và trade-off giữa độ chính xác (mAP50) và tốc độ suy luận (FPS/Latency) trên toàn bộ 6 subtest.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path
from pprint import pprint

candidate = Path.cwd().resolve()
for directory in (candidate, *candidate.parents):
    if (directory / 'pyproject.toml').is_file():
        PROJECT_ROOT = directory
        break
else:
    raise RuntimeError('Không tìm thấy project root (pyproject.toml).')

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

E5_CONFIG = PROJECT_ROOT / 'configs' / 'E5_full_method.yaml'
E2_CHECKPOINT = PROJECT_ROOT / 'experiments' / 'E2_augmentation' / 'augmentation_b16_seed42' / 'weights' / 'best.pt'
print(f'E5 Config: {E5_CONFIG}')
print(f'E2 Checkpoint: {E2_CHECKPOINT} (Exists: {E2_CHECKPOINT.is_file()})')

## 1. Tải cấu hình và khởi tạo mô hình


In [ ]:
from helmet_yolov10.utils.config import load_config
from helmet_yolov10.training.train import _load_backend
from helmet_yolov10.inference.full_method import predict_full_method

config = load_config(E5_CONFIG)
pprint(config)

backend_cls = _load_backend()
model = backend_cls(str(E2_CHECKPOINT))
print('✅ Đã load mô hình E2 thành công cho E5 pipeline.')

## 2. Demo suy luận trên 1 ảnh ví dụ


In [ ]:
import cv2
import matplotlib.pyplot as plt

IMAGE_PATH = PROJECT_ROOT / 'data' / 'processed' / 'images' / 'val' / 'example.jpg' # Thay đường dẫn ảnh của bạn

if not IMAGE_PATH.is_file():
    print(f'⚠️ Hãy chọn một đường dẫn ảnh thực tế để chạy demo (hiện tại chưa tìm thấy {IMAGE_PATH})')
else:
    image = cv2.imread(str(IMAGE_PATH))
    if image is not None:
        res = predict_full_method(
            model=model,
            image=image,
            enhancement_config=config.get('enhancement'),
            tiled_config=config.get('tiled_inference'),
            device=0
        )
        print(f"Số box phát hiện: {len(res['boxes'])}, Thời gian: {res['total_seconds']*1000:.1f}ms, FPS: {res['total_fps']:.1f}")